# Week 1 — Linear algebra, norms, convolution & Fourier (the math foundations)

This is the hands-on companion for the **"mathematical foundations"** half of
Week 1 (the probability half is in `02_probability_refresher.ipynb`). Everything
here is built from scratch with **NumPy + Matplotlib only** — no other libraries.

By the end you will have *coded*, not just read:

1. An image as a **vector**, and the **L1 / L2 / L∞ norms** (and why L1 → sparsity).
2. **Inner products**, angles, orthogonality.
3. A **matrix as a linear map** (circle → ellipse), transpose, inverse.
4. **Eigenvectors / eigenvalues**, **positive-definite** matrices, and the **SVD**
   (as low-rank image compression).
5. **2D convolution by hand**, confirmed against NumPy (Readme §1, goal #3).
6. The **2D DFT** of a delta and a sinusoid, and the **convolution theorem**
   (Readme §6, exercise 5).

> Companion reading: the `handbook_main.pdf` Week 1 chapter; for where this math
> reappears in the decks, see `Slides_MIC_ImageModeling.pdf` (norms / ridge / Lasso,
> slides 69–84) and `Slides_MIC_XrayCT.pdf` (Fourier & the Central Slice Theorem,
> slides 61–72).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)
rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (10, 4)
print('numpy', np.__version__)

## 1. An image is a vector — and norms measure its "size"

Stack the rows of an `H×W` image and you get a length-`HW` **vector**. Almost
every algorithm in this course treats images exactly this way. To measure the
size of (or distance between) such vectors we use **norms**:

- **L2** (Euclidean): \(\|x\|_2=\sqrt{\sum_i x_i^2}\) — ordinary length.
- **L1** (Manhattan): \(\|x\|_1=\sum_i |x_i|\) — promotes **sparsity**.
- **L∞** (max): \(\|x\|_\infty=\max_i |x_i|\).

The unit ball \(\{x:\|x\|\le 1\}\) has a different shape for each norm, and that
geometry is *why* L1 produces sparse solutions — a fact we lean on for
total-variation denoising and compressed sensing in Week 3.

In [ ]:
# --- an image IS a vector: just reshape it ---
img = np.array([[10, 20, 30],
                [40, 50, 60],
                [70, 80, 90]], dtype=float)
v = img.reshape(-1)                 # flatten 3x3 image -> length-9 vector
print('image shape', img.shape, '-> vector shape', v.shape)
print('back to image:\n', v.reshape(3, 3))

# --- the three norms you must know ---
x = np.array([3.0, -4.0])
l1   = np.sum(np.abs(x))            # = 7
l2   = np.sqrt(np.sum(x**2))        # = 5  (Euclidean length)
linf = np.max(np.abs(x))           # = 4
print(f'\nx = {x}:   L1={l1:.1f}  L2={l2:.1f}  Linf={linf:.1f}')
print('numpy check :',
      np.linalg.norm(x, 1), np.linalg.norm(x, 2), np.linalg.norm(x, np.inf))

# --- geometry: the unit ball changes shape per norm ---
theta = np.linspace(0, 2*np.pi, 400)
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))

ax[0].plot(np.cos(theta), np.sin(theta), label='L2 (circle)')
ax[0].plot([1, 0, -1, 0, 1], [0, 1, 0, -1, 0], label='L1 (diamond)')
ax[0].plot([1, -1, -1, 1, 1], [1, 1, -1, -1, 1], label='Linf (square)')
ax[0].set_aspect('equal'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[0].set_title('Unit balls  {x : ||x|| = 1}')

# --- why L1 promotes sparsity: its prox (soft-threshold) zeroes small values ---
t = 1.0
u = np.linspace(-4, 4, 400)
soft  = np.sign(u) * np.maximum(np.abs(u) - t, 0.0)   # L1 prox -> EXACT zeros
ridge = u / (1 + t)                                   # L2 prox -> only shrinks
ax[1].plot(u, u, 'k--', alpha=.4, label='identity')
ax[1].plot(u, soft,  label='L1 prox (soft-threshold)')
ax[1].plot(u, ridge, label='L2 prox (ridge shrink)')
ax[1].axhline(0, color='gray', lw=.5); ax[1].axvline(0, color='gray', lw=.5)
ax[1].legend(); ax[1].grid(alpha=.3)
ax[1].set_title('L1 sets small coefficients to EXACTLY 0 (sparsity)')
plt.tight_layout(); plt.show()

## 2. Inner products and matrices as linear maps

The **inner product** \(\langle x,y\rangle = x^\top y\) gives lengths
(\(\|x\|=\sqrt{\langle x,x\rangle}\)) and angles
(\(\cos\theta = \langle x,y\rangle/(\|x\|\,\|y\|)\)); it is **0** exactly when two
vectors are orthogonal. A **matrix is a linear map**: multiplying by `A` sends the
unit circle to an ellipse, and `inv(A)` sends it back.

In [ ]:
# --- inner product, length, angle ---
a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, 5.0, 6.0])
dot = a @ b                                   # inner product = sum of products
cos = dot / (np.linalg.norm(a) * np.linalg.norm(b))
print('a.b =', dot, '   cos(angle) =', round(cos, 3),
      '   angle(deg) =', round(np.degrees(np.arccos(cos)), 2))

e1, e2 = np.array([1.0, 0.0]), np.array([0.0, 1.0])
print('orthogonal example: e1.e2 =', e1 @ e2, '(0 => perpendicular)')

# --- a matrix is a linear map: it sends the unit circle to an ellipse ---
A = np.array([[2.0, 1.0],
              [1.0, 2.0]])
theta = np.linspace(0, 2*np.pi, 200)
circle  = np.vstack([np.cos(theta), np.sin(theta)])   # 2 x 200 unit circle
ellipse = A @ circle                                  # mapped points
back    = np.linalg.inv(A) @ ellipse                  # inverse maps it back

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
ax[0].plot(circle[0], circle[1], label='unit circle')
ax[0].plot(ellipse[0], ellipse[1], label='A @ circle (ellipse)')
ax[0].set_aspect('equal'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[0].set_title('A matrix maps the circle to an ellipse')

ax[1].plot(ellipse[0], ellipse[1], label='ellipse')
ax[1].plot(back[0], back[1], '--', label='inv(A) @ ellipse = circle again')
ax[1].set_aspect('equal'); ax[1].legend(); ax[1].grid(alpha=.3)
ax[1].set_title('inv(A) undoes the map')
plt.tight_layout(); plt.show()

print('A^T (transpose):\n', A.T)
print('A is symmetric:', np.allclose(A, A.T))

## 3. Eigenvectors, positive-definite matrices, and the SVD

For a **symmetric** matrix `A`, the eigen-decomposition \(Av=\lambda v\) has real
eigenvalues and orthogonal eigenvectors — they are the **axes** of the ellipse
\(x^\top A x = 1\). `A` is **positive-definite** when \(x^\top A x>0\) for all
\(x\neq 0\) (equivalently: all eigenvalues > 0; equivalently: Cholesky succeeds);
covariance and kernel matrices are positive (semi)definite.

The **SVD** \(A = U\Sigma V^\top\) is the workhorse: keeping only the largest few
singular values gives the best low-rank approximation — i.e. **image compression**,
and the basis for PCA (Week 4).

In [ ]:
# --- eigen-decomposition of a SYMMETRIC matrix (real eigenvalues, orthogonal eigenvectors) ---
A = np.array([[2.0, 1.0],
              [1.0, 2.0]])
w, V = np.linalg.eigh(A)            # eigh: for symmetric/Hermitian matrices
print('eigenvalues :', w)
print('eigenvectors (columns):\n', V)
print('eigenvectors orthogonal: v0.v1 =', round(V[:, 0] @ V[:, 1], 12))

# the ellipse x^T A x = 1 has axes ALONG the eigenvectors, semi-length 1/sqrt(eigenvalue)
theta = np.linspace(0, 2*np.pi, 300)
unit = np.vstack([np.cos(theta), np.sin(theta)])
pts = V @ np.diag(1/np.sqrt(w)) @ unit

fig, ax = plt.subplots(1, 3, figsize=(14, 4.2))
ax[0].plot(pts[0], pts[1])
for k in range(2):
    vec = V[:, k] / np.sqrt(w[k])
    ax[0].arrow(0, 0, vec[0], vec[1], head_width=.05, color='r',
                length_includes_head=True)
ax[0].set_aspect('equal'); ax[0].grid(alpha=.3)
ax[0].set_title('x^T A x = 1: eigenvectors are the axes')

# --- positive-definite check: all eigenvalues > 0  <=>  Cholesky succeeds ---
def is_pd(M):
    if not np.allclose(M, M.T):
        return False
    try:
        np.linalg.cholesky(M)
        return True
    except np.linalg.LinAlgError:
        return False

B_pd  = np.array([[2.0, 1.0], [1.0, 2.0]])     # eigenvalues 1, 3  -> PD
B_not = np.array([[0.0, 1.0], [1.0, 0.0]])     # eigenvalues -1, 1 -> not PD
print('\nB_pd  eigenvalues', np.linalg.eigvalsh(B_pd),  '  is_pd =', is_pd(B_pd))
print('B_not eigenvalues', np.linalg.eigvalsh(B_not), '  is_pd =', is_pd(B_not))

# --- SVD: A = U S V^T.  Low-rank truncation = image compression ---
N = 100
yy, xx = np.mgrid[0:N, 0:N]
image = 0.5 + 0.5*np.sin(2*np.pi*xx/22)        # stripes (low rank)
image[(xx-50)**2 + (yy-50)**2 < 18**2] = 1.0   # + a disk (adds rank)

U, S, Vt = np.linalg.svd(image, full_matrices=False)
def low_rank(k):
    return (U[:, :k] * S[:k]) @ Vt[:k]         # best rank-k approximation

ax[1].imshow(image, cmap='gray'); ax[1].set_title('original'); ax[1].axis('off')
ax[2].semilogy(S + 1e-12); ax[2].set_title('singular values (energy per mode)')
ax[2].set_xlabel('index'); ax[2].grid(alpha=.3)
plt.tight_layout(); plt.show()

ks = [1, 3, 10, 40]
fig, axes = plt.subplots(1, len(ks), figsize=(14, 3.4))
for a_, k in zip(axes, ks):
    rec = low_rank(k)
    err = np.linalg.norm(image - rec) / np.linalg.norm(image)
    a_.imshow(rec, cmap='gray'); a_.axis('off')
    a_.set_title(f'rank {k}\nrel.err {err:.2f}')
plt.tight_layout(); plt.show()

## 4. Convolution — by hand, then confirmed with NumPy

**Convolution** is the core image operation:
\((I*k)[m,n]=\sum_{i,j} I[m-i,\,n-j]\,k[i,j]\) — note the kernel is **flipped**
(that is the difference from *correlation*, which most "filtering" actually uses).
We implement it straight from the definition, cross-check it with NumPy's FFT,
verify the hand-worked 3×3 example (`assert`), then reuse the same machinery for
mean / Gaussian / Sobel filters.

In [ ]:
# --- 2D convolution BY HAND (full), then confirm with NumPy's FFT ---
def conv2d_full(I, k):
    """Full 2D convolution from the definition (kernel flipped via i-a, j-b)."""
    H1, W1 = I.shape
    H2, W2 = k.shape
    H, W = H1 + H2 - 1, W1 + W2 - 1
    out = np.zeros((H, W))
    for i in range(H):
        for j in range(W):
            s = 0.0
            for a in range(H2):
                for bb in range(W2):
                    ii, jj = i - a, j - bb
                    if 0 <= ii < H1 and 0 <= jj < W1:
                        s += I[ii, jj] * k[a, bb]
            out[i, j] = s
    return out

def conv2d_fft(I, k):
    """Same full convolution via the FFT (NumPy only) -- our cross-check."""
    H = I.shape[0] + k.shape[0] - 1
    W = I.shape[1] + k.shape[1] - 1
    F = np.fft.fft2(I, s=(H, W)) * np.fft.fft2(k, s=(H, W))
    return np.real(np.fft.ifft2(F))

I = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]], dtype=float)
k = np.array([[1, 0, -1],
              [1, 0, -1],
              [1, 0, -1]], dtype=float)

hand = conv2d_full(I, k)
print('hand-rolled == NumPy-FFT :', np.allclose(hand, conv2d_fft(I, k)))   # True

# the 'valid' (no padding) output of two 3x3 arrays is the single centre value:
valid = hand[2, 2]
print('valid 3x3 convolution result =', valid, '(hand-computed answer: 6)')
assert np.isclose(valid, 6.0)

# CONVOLUTION (flips kernel) vs CORRELATION (no flip) differ for asymmetric kernels:
print('correlation (no flip)        =', np.sum(I * k), '(= -6, opposite sign here)')

# --- the same machinery = image filtering ---
def filter2d(I, k, mode='conv'):
    """Same-size filtering with zero padding. mode='conv' flips the kernel."""
    kk = k[::-1, ::-1] if mode == 'conv' else k
    kh, kw = kk.shape
    ph, pw = kh // 2, kw // 2
    P = np.pad(I, ((ph, ph), (pw, pw)))
    out = np.zeros_like(I, dtype=float)
    for i in range(I.shape[0]):
        for j in range(I.shape[1]):
            out[i, j] = np.sum(P[i:i+kh, j:j+kw] * kk)
    return out

def gaussian_kernel(size=5, sigma=1.5):
    ax = np.arange(size) - size // 2
    gx, gy = np.meshgrid(ax, ax)
    g = np.exp(-(gx**2 + gy**2) / (2 * sigma**2))
    return g / g.sum()

# a small synthetic test image (numpy only)
N = 80
yy, xx = np.mgrid[0:N, 0:N]
test = np.zeros((N, N))
test[(xx-28)**2 + (yy-28)**2 < 16**2] = 1.0
test[50:70, 48:72] = 0.6

mean_k  = np.ones((5, 5)) / 25.0
gauss_k = gaussian_kernel(5, 1.5)
sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=float)

outs = {'original': test,
        'mean 5x5': filter2d(test, mean_k),
        'gaussian 5x5': filter2d(test, gauss_k),
        '|Sobel-x| (edges)': np.abs(filter2d(test, sobel_x))}
fig, axes = plt.subplots(1, 4, figsize=(14, 3.6))
for a_, (title, im) in zip(axes, outs.items()):
    a_.imshow(im, cmap='gray'); a_.set_title(title); a_.axis('off')
plt.tight_layout(); plt.show()

## 5. The 2D DFT and the convolution theorem

The **2D DFT** rewrites an image as a sum of 2D sinusoids (spatial frequencies).
Two instructive cases:

- a **delta** (single bright point) contains **every** frequency equally → a flat
  spectrum;
- a **pure sinusoid** contains a single frequency → **two** symmetric peaks.

The **convolution theorem** then says convolution in space = **pointwise
multiplication** in frequency. This is why MRI (acquired in frequency / *k*-space,
see `Slides_MIC_MRI.pdf` slide 50) and CT reconstruction (`Slides_MIC_XrayCT.pdf`
slides 61–72) are naturally described in the Fourier domain.

In [ ]:
# --- 2D DFT of a single delta (point) and a single sinusoid ---
N = 64
delta = np.zeros((N, N)); delta[N//2, N//2] = 1.0

yy, xx = np.mgrid[0:N, 0:N]
freq = 6
sinus = np.sin(2*np.pi*freq*xx/N)              # horizontal sinusoid

def logmag(im):
    return np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(im))))

fig, ax = plt.subplots(2, 2, figsize=(9, 8))
ax[0, 0].imshow(delta, cmap='gray'); ax[0, 0].axis('off')
ax[0, 0].set_title('delta (a single point)')
ax[0, 1].imshow(logmag(delta), cmap='magma'); ax[0, 1].axis('off')
ax[0, 1].set_title('its spectrum: FLAT (all frequencies)')
ax[1, 0].imshow(sinus, cmap='gray'); ax[1, 0].axis('off')
ax[1, 0].set_title(f'sinusoid (freq={freq})')
ax[1, 1].imshow(logmag(sinus), cmap='magma'); ax[1, 1].axis('off')
ax[1, 1].set_title('its spectrum: TWO symmetric peaks')
plt.tight_layout(); plt.show()

# --- convolution theorem: convolution in space == multiplication in frequency ---
img = test                                     # reuse test image + gaussian_kernel from §4
ker = gaussian_kernel(5, 1.5)
space = conv2d_full(img, ker)
H, W = space.shape
freq_domain = np.real(np.fft.ifft2(
    np.fft.fft2(img, s=(H, W)) * np.fft.fft2(ker, s=(H, W))))
print('convolution theorem holds (space-conv == freq-product):',
      np.allclose(space, freq_domain))

## Exercises

1. Use `sobel_x.T` to detect **horizontal** edges, then combine the two with
   `np.hypot(gx, gy)` to get gradient magnitude.
2. Make a noisy *sparse* signal and recover it with the soft-threshold (L1) vs.
   ridge shrink (L2). Which one restores the exact zeros?
3. Take the SVD of any real grayscale array and plot relative error vs. rank `k`.
   How many modes until it looks "visually lossless"?
4. Verify the convolution theorem for the (asymmetric) Sobel kernel — mind the flip.
5. Compute the 2D DFT of a *rotated* sinusoid and predict where the peaks move.

## What's next

Week 2 uses all of this immediately: images-as-vectors with quadratic (L2) vs.
sparse (L1) penalties become **priors**, convolution becomes the **blur /
degradation model**, and the Fourier view returns for CT and MRI in Week 3.
See `02_probability_refresher.ipynb` for the probability half (MLE vs MAP), and
the handbook's Week 1 chapter for the written treatment.